# Gold Layer: Business Aggregates for Power BI

This notebook creates business-level aggregates and final tables for Power BI consumption.

**Tasks:**
- Join silver tables to create facts and dimensions
- Calculate KPIs and metrics
- Optimize for Power BI DirectQuery
- Apply final business logic
- Write gold Delta tables

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, round, date_trunc, month, year

spark = SparkSession.builder \
    .appName("GoldAggregation") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

## Load Silver Tables

In [ ]:
silver_orders = spark.read.format("delta").load("/mnt/delta/silver/orders")
silver_customers = spark.read.format("delta").load("/mnt/delta/silver/customers")
silver_products = spark.read.format("delta").load("/mnt/delta/silver/products")

# Load order items - assuming we have a bronze order_items that gets silver processed
try:
    silver_order_items = spark.read.format("delta").load("/mnt/delta/silver/order_items")
except:
    # If order_items doesn't exist, we'll create it from bronze during processing
    silver_order_items = None

## Dimension: Customer

In [ ]:
dim_customer = silver_customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "city",
    "state",
    "customer_segment",
    "signup_date"
).distinct()

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/gold/dim_customer")

## Dimension: Product

In [ ]:
dim_product = silver_products.select(
    "product_id",
    "product_name",
    "category",
    "price",
    "cost",
    "inventory_quantity"
).distinct()

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/gold/dim_product")

## Fact: Sales

Create fact table with order and order items joined with dimensions

In [ ]:
# If we have order items, create detailed fact
if silver_order_items:
    fact_sales = silver_order_items \
        .join(silver_orders, "order_id") \
        .join(dim_customer, "customer_id") \
        .join(dim_product, "product_id") \
        .select(
            "order_id",
            "customer_id",
            "product_id",
            "order_date",
            "status",
            "quantity",
            "unit_price",
            "line_total",
            "customer_segment",
            "category"
        )
else:
    # Aggregate order totals if no order items
    fact_sales = silver_orders \
        .join(dim_customer, "customer_id") \
        .select(
            "order_id",
            "customer_id",
            "order_date",
            "total_amount",
            "status",
            "customer_segment"
        )

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/gold/fact_sales")

## KPI Aggregates

Pre-compute common KPIs for Power BI

In [ ]:
# Monthly sales summary
monthly_sales = fact_sales \
    .withColumn("year_month", date_trunc("month", col("order_date"))) \
    .groupBy("year_month") \
    .agg(
        sum("line_total") if "line_total" in fact_sales.columns else sum("total_amount"),
        count("order_id").alias("order_count"),
        avg("line_total").alias("avg_order_value") if "line_total" in fact_sales.columns else avg("total_amount").alias("avg_order_value")
    ) \
    .withColumn("total_sales", round(col("sum(line_total)"), 2) if "line_total" in fact_sales.columns else round(col("sum(total_amount)"), 2))

monthly_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/gold/monthly_sales")

## Verification

In [ ]:
print("Gold tables created:")
print(f"Dim Customer: {spark.read.format('delta').load('/mnt/delta/gold/dim_customer').count()} rows")
print(f"Dim Product: {spark.read.format('delta').load('/mnt/delta/gold/dim_product').count()} rows")
print(f"Fact Sales: {spark.read.format('delta').load('/mnt/delta/gold/fact_sales').count()} rows")
print(f"Monthly Sales: {spark.read.format('delta').load('/mnt/delta/gold/monthly_sales').count()} rows")